In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def filter_rmats_dev(infile,outfile,newoutfile,listsamples,FDRval,deltaval,mean_cov,filter_events):
    # Load the CSV file
    file_path = infile
    data = pd.read_csv(file_path, delimiter='\t')
    file = open(file_path,'r')
    
    COLUMN_NAMES = listsamples
    out_F = 0
    in_F = 0
    sig_genes = {}
    sig_genes_key = {}
    
    df = pd.DataFrame(columns=COLUMN_NAMES)

    events = []
    a = 0
    for i in file:
        a+=1
        if a == 1:
            #output.write(i)
            continue
        else:
            field = i.split('\t')
            IJC = [int(x) for x in field[12].split(',')]
            SJC = [int(x) for x in field[13].split(',')]
            IJC2 = [int(x) for x in field[14].split(',')]
            SJC2 = [int(x) for x in field[15].split(',')]
            if 'NA' in field[20] or 'NA' in field[21]:
                continue
            psi1 = [float(x) for x in field[20].split(',') if x != 'NA']
            psi2 = [float(x) for x in field[21].split(',') if x != 'NA']
            psitot = psi1 + psi2
            geneSymbol = field[2].replace('\"','')
            key = field[3]+'-'+field[5]+'-'+field[6]+'-'+field[7]+'-'+field[8]+'-'+field[9]+'-'+field[10]
                          
            if filter_events:
                if key in filter_events:
                    df.loc[geneSymbol] = psitot

    file.close()

    return df

def PCA_df_dev(df,listsamples,shownames,outsvg,FDR,delta):
    
    df_transposed = df.transpose()
    
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(df_transposed)
    
    y = []
    for x in listsamples:
        if 'CTR' in x:
            y.append(0)
        elif 'RNU' in x:
            y.append(1)
        else:
            y.append(2)
    
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], c=y, edgecolor='k', s=50)
    # Annotating each point
    if shownames:
        for i, sample_name in enumerate(listsamples):
            if not 'CTR' in sample_name:
                plt.annotate(sample_name, (pca_result[i, 0], pca_result[i, 1]))
    
    #ax = plt.gca()
    #ax.set_xlim([-1, 1])
    #ax.set_ylim([-1, 1])
    
    plt.title('PCA of PSI values with FDR<'+str(FDR)+' and |deltaPSI|>'+str(delta)+', n='+str(len(df_transposed.columns)))
    plt.xlabel('Principal Component 1: '+str(int(float(pca.explained_variance_ratio_[0])*100))+'%')
    plt.ylabel('Principal Component 2: '+str(int(float(pca.explained_variance_ratio_[1])*100))+'%')
    #plt.colorbar(label='Sample Type (0=Unaffected, 1=Affected)')
    
    plt.grid(True)
    if outsvg:
        plt.savefig(outsvg, format="pdf", transparent=True)
    
    plt.show()

    #print feattures based on importance PC-1
    #pca_features_df = pd.DataFrame(pca.components_,columns=df_transposed.columns,index = ['PC-1','PC-2'])
    #abs_values = pca_features_df.iloc[0].abs()
    #sorted_columns = abs_values.sort_values(ascending=False).index
    #sorted_pca_features_df = pca_features_df[sorted_columns]
    #print(sorted_pca_features_df.iloc[:, :10])

    #print feattures based on importance PC-2
    #abs_values_pc2 = pca_features_df.iloc[1].abs()
    #sorted_columns_pc2 = abs_values_pc2.sort_values(ascending=False).index
    #sorted_pca_features_pc2_df = pca_features_df[sorted_columns_pc2]
    #print(sorted_pca_features_pc2_df.iloc[:, :10])
